In [109]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [110]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

In [111]:
air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

In [112]:
air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

In [113]:
dataset = pd.concat([weather, air_qual], axis=1)
dataset.drop(columns=["time"], inplace=True)

# setting up the graph

In [114]:
import networkx as nx
import torch
from torch_geometric.utils import to_networkx
from torch_geometric.nn import GCNConv
from torch.nn import Linear
import geopy.distance
from torch_geometric.data import Data


In [115]:
G = nx.DiGraph()

for sensor_id, directions in traffic_dic.items():
    for direction, df in directions.items():
        origin_lat = df["latitude"].iloc[0]
        origin_lon = df["longitude"].iloc[0]

        G.add_node(sensor_id, pos=(origin_lon, origin_lat))
        
        for sensor_id_2, directions_2 in traffic_dic.items():
            if sensor_id == sensor_id_2:
                continue

            for direction_2, df_2 in directions_2.items():
                target_lat = df_2["latitude"].iloc[0]
                target_lon = df_2["longitude"].iloc[0]
                
                distance_m = geopy.distance.geodesic(
                    (origin_lat, origin_lon), 
                    (target_lat, target_lon)
                ).m

                if distance_m < 1000:
                    G.add_edge(
                        sensor_id, 
                        sensor_id_2, 
                        distance=distance_m, 
                        dir=direction
                    )

In [116]:
manual_edges = [(11,23), (9,22), (9,23), (1,12), (2,14), (3,14),(4,14), (5,15), (8,20), (6,19), (6,16), (8,19), (7,20), (7,19), (6,5), (17,14), (15,14), (16,14)]

In [117]:
import geopy.distance

node_positions = nx.get_node_attributes(G, "pos")

for source, target in manual_edges:
    lon1, lat1 = node_positions[source]
    lon2, lat2 = node_positions[target]

    distance_m = geopy.distance.geodesic((lat1, lon1), (lat2, lon2)).m
    G.add_edge(source, target, distance=distance_m)

In [118]:
sensor_ids = list(G.nodes())
node_map = {sid: idx for idx, sid in enumerate(sensor_ids)}
num_nodes = len(sensor_ids)

sample_df = list(traffic_dic[sensor_ids[0]].values())[0]

exclude_cols = ['latitude', 'longitude', 'timestamp', 'time', 'miles', "avtime", "hour"]
base_features = [col for col in sample_df.columns if col not in exclude_cols]

time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos']
all_features = base_features + time_features

num_timestamps = len(sample_df)
num_features = len(all_features)

x_tensor = torch.zeros((num_nodes, num_features, num_timestamps), dtype=torch.float)

for sensor_id in sensor_ids:
    node_idx = node_map[sensor_id]
    traffic_df = list(traffic_dic[sensor_id].values())[0].copy()
    
    traffic_df['timestamp'] = pd.to_datetime(traffic_df['timestamp'])
    
    traffic_df['hour_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['hour_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['day_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df['day_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df["count_imputed"] = traffic_df["count_imputed"].apply(lambda x: 0 if x == False else 1)
    traffic_features_matrix = traffic_df[all_features].values.T
    x_tensor[node_idx, :, :] = torch.tensor(traffic_features_matrix, dtype=torch.float)

edge_list = []
edge_attr_list = []
for u, v, data in G.edges(data=True):
    edge_list.append([node_map[u], node_map[v]])
    edge_attr_list.append([data['distance']])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)

pyg_dataset = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr)

In [119]:
import plotly.graph_objects as go

num_nodes = len(node_map)
coords_traffic = np.zeros((num_nodes, 2))

for sensor_id, node_idx in node_map.items():
    lon, lat = G.nodes[sensor_id]['pos']
    coords_traffic[node_idx, 0] = lon
    coords_traffic[node_idx, 1] = lat

edge_t_to_t = pyg_dataset.edge_index


def visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t, mapbox_style="carto-positron"):
    """
    Visualizes the traffic sensor spatial graph directly on a map using Plotly Scattermapbox.
    """
    fig = go.Figure()

    # --- DRAW TRAFFIC-TO-TRAFFIC EDGES ---
    t_edge_lon, t_edge_lat = [], []
    t_start_nodes = edge_t_to_t[0].numpy()
    t_end_nodes = edge_t_to_t[1].numpy()
    
    for src, dst in zip(t_start_nodes, t_end_nodes):
        # Plotly draws continuous paths; adding None breaks the line between distinct edges
        t_edge_lon.extend([coords_traffic[src, 0], coords_traffic[dst, 0], None])
        t_edge_lat.extend([coords_traffic[src, 1], coords_traffic[dst, 1], None])
        
    fig.add_trace(go.Scattermapbox(
        lon=t_edge_lon, lat=t_edge_lat,
        mode='lines',
        line=dict(width=1.5, color='rgba(50, 150, 250, 0.6)'),
        name='Traffic-to-Traffic Edges',
        hoverinfo='none'
    ))

    fig.add_trace(go.Scattermapbox(
        lon=coords_traffic[:, 0], lat=coords_traffic[:, 1],
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.85),
        name='Traffic Sensor Nodes',
        text=[f"Node Index: {i}<br>Sensor ID: {sensor_ids[i]}" for i in range(len(coords_traffic))],
        hoverinfo='text'
    ))

    center_lat = np.mean(coords_traffic[:, 1])
    center_lon = np.mean(coords_traffic[:, 0])

    fig.update_layout(
        title=dict(text='Spatio-Temporal Traffic Graph Topology', font=dict(size=18)),
        autosize=True,
        hovermode='closest',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.7)"),
        mapbox=dict(
            style=mapbox_style,
            bearing=0,
            center=dict(lat=center_lat, lon=center_lon),
            pitch=0,
            zoom=12
        ),
        width=1100,
        height=750,
        margin=dict(r=0, t=40, l=0, b=0)
    )
    
    fig.show()

visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t)

/tmp/ipykernel_3375486/3245259071.py:30: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/tmp/ipykernel_3375486/3245259071.py:38: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


In [120]:
temporal_df.columns

Index(['time_x', 'temperature_2m (°C)', 'relative_humidity_2m (%)',
       'dew_point_2m (°C)', 'apparent_temperature (°C)', 'precipitation (mm)',
       'rain (mm)', 'weather_code (wmo code)', 'wind_speed_10m (km/h)',
       'wind_speed_100m (km/h)', 'wind_direction_10m (°)',
       'wind_direction_100m (°)', 'wind_gusts_10m (km/h)',
       'cloud_cover_high (%)', 'cloud_cover_mid (%)', 'cloud_cover_low (%)',
       'cloud_cover (%)', 'surface_pressure (hPa)', 'time_y', 'pm10 (μg/m³)',
       'pm2_5 (μg/m³)', 'nitrogen_dioxide (μg/m³)', 'sulphur_dioxide (μg/m³)',
       'carbon_monoxide (μg/m³)', 'ozone (μg/m³)', 'hour_sin', 'hour_cos'],
      dtype='object')

In [121]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. GRAPH STRUCTURE + TEMPORAL FEATURE PREPARATION
# ==========================================
# Extract dimensions from the existing graph tensor
num_nodes_spatial = pyg_dataset.x.shape[0]
num_features_spatial = pyg_dataset.x.shape[1]
total_timestamps = pyg_dataset.x.shape[2]

# Build the spatial adjacency matrix from the PyG edge index
adj_matrix = np.zeros((num_nodes_spatial, num_nodes_spatial), dtype=np.float32)
edges = pyg_dataset.edge_index.numpy()
adj_matrix[edges[0], edges[1]] = 1.0
adj_matrix += np.eye(num_nodes_spatial, dtype=np.float32)

deg = np.sum(adj_matrix, axis=1)
deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
deg_inv_sqrt[deg == 0] = 0.0
D_inv_sqrt = np.diag(deg_inv_sqrt)
normalized_adj = D_inv_sqrt @ adj_matrix @ D_inv_sqrt

# Convert traffic tensor to a time-major layout for windowing
traffic_np = pyg_dataset.x.numpy()
traffic_time_major = np.transpose(traffic_np, (2, 0, 1))  # (timestamps, nodes, features)

temporal_df = pd.merge(weather, air_qual, left_on="timestamp", right_on="time", how="inner")
temporal_df = temporal_df.sort_values("timestamp").set_index("timestamp")
temporal_df = temporal_df.asfreq("h")
temporal_df = temporal_df.drop(columns=["time"], errors="ignore")
idx_hour = temporal_df.index.hour
temporal_df["hour_sin"] = np.sin(2 * np.pi * idx_hour / 24.0)
temporal_df["hour_cos"] = np.cos(2 * np.pi * idx_hour / 24.0)
temporal_df = temporal_df.dropna()

temporal_feature_cols = [
    "hour_sin",
    "hour_cos",
    "temperature_2m (°C)",
    "surface_pressure (hPa)",
    "wind_speed_100m (km/h)",
    "precipitation (mm)",
]
temporal_feature_cols = [col for col in temporal_feature_cols if col in temporal_df.columns]
target_col = "pm2_5 (μg/m³)"

scaler_X = MinMaxScaler()
scaled_temporal = scaler_X.fit_transform(temporal_df[temporal_feature_cols])
target_series = temporal_df[target_col].to_numpy(dtype=np.float32)

# Optional extra branch for weather-only features; keep it disabled by default.
ADDITIONAL_BRANCH = False
if ADDITIONAL_BRANCH:
    weather_feature_cols = ['pm10 (μg/m³)', 'nitrogen_dioxide (μg/m³)', 'carbon_monoxide (μg/m³)', 'ozone (μg/m³)']
    weather_feature_cols = [col for col in weather_feature_cols if col in temporal_df.columns]
    scaler_X_weather = MinMaxScaler()
    scaled_weather = scaler_X_weather.fit_transform(temporal_df[weather_feature_cols])
else:
    scaled_weather = None

# Sliding-window generation for the two main branches
lookback = 24 * 3
forecast_steps = 24

traffic_timestamps = pd.to_datetime(temporal_df.index[:traffic_time_major.shape[0]])
traffic_series_map = pd.Series(range(len(traffic_timestamps)), index=traffic_timestamps)

x_traffic = []
x_temporal = []
x_weather_branch = []
y = []

for i in range(lookback, len(temporal_df) - forecast_steps + 1):
    current_time_window = temporal_df.index[i - lookback : i]
    try:
        traffic_indices = traffic_series_map.loc[current_time_window].values.astype(int)
        if len(traffic_indices) != lookback or np.isnan(traffic_indices).any():
            continue

        traffic_slice = traffic_time_major[traffic_indices, :, :]
        if traffic_slice.shape != (lookback, num_nodes_spatial, num_features_spatial):
            continue

        x_traffic.append(traffic_slice)
        x_temporal.append(scaled_temporal[i - lookback : i, :])
        if ADDITIONAL_BRANCH:
            x_weather_branch.append(scaled_weather[i - lookback : i, :])
        y.append(target_series[i : i + forecast_steps])
    except KeyError:
        continue

x_traffic = np.array(x_traffic, dtype=np.float32)
x_temporal = np.array(x_temporal, dtype=np.float32)
y = np.array(y, dtype=np.float32)

if ADDITIONAL_BRANCH:
    x_weather_branch = np.array(x_weather_branch, dtype=np.float32)

print(f"Traffic branch shape: {x_traffic.shape}")
print(f"Temporal branch shape: {x_temporal.shape}")
print(f"Target shape: {y.shape}")
if ADDITIONAL_BRANCH:
    print(f"Weather branch shape: {x_weather_branch.shape}")

# Train/Test split
split = int(len(y) * 0.7)
x_train_traffic, x_test_traffic = x_traffic[:split], x_traffic[split:]
x_train_temporal, x_test_temporal = x_temporal[:split], x_temporal[split:]
y_train, y_test = y[:split], y[split:]

if ADDITIONAL_BRANCH:
    x_train_weather, x_test_weather = x_weather_branch[:split], x_weather_branch[split:]
else:
    x_train_weather = None
    x_test_weather = None

# Repeat the adjacency for each sample in the dataset
adj_train = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_train_traffic), axis=0)
adj_test = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_test_traffic), axis=0)

num_temporal_features = x_temporal.shape[-1]


Traffic branch shape: (6879, 72, 24, 11)
Temporal branch shape: (6879, 72, 6)
Target shape: (6879, 24)


In [122]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv

class TransformerLSTMModel(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_transformer_layers, hidden_dim, num_lstm_layers, output_dim, seq_len, dropout=0.1):
        super(TransformerLSTMModel, self).__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        self.lstm = nn.LSTM(
            input_size=d_model, hidden_size=hidden_dim, num_layers=num_lstm_layers,
            batch_first=True, dropout=dropout if num_lstm_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        x = self.input_projection(src)
        x = x + self.pos_encoder[:, :src.size(1), :]
        x = self.dropout(x)
        transformer_out = self.transformer_encoder(x)
        lstm_out, (hn, cn) = self.lstm(transformer_out)
        out = lstm_out[:, -1, :]
        predictions = self.fc(self.dropout(out))
        return predictions


class TriBranchSpatioTemporalModel(nn.Module):
    def __init__(self, num_nodes, traffic_feat_dim, weather_feat_dim, seq_len, output_dim, 
                 transformer_kwargs, graph_hidden_dim=64, fusion_hidden_dim=128, dropout=0.2, use_weather=True):
        super().__init__()
        self.num_nodes = num_nodes
        self.output_dim = output_dim
        self.use_weather = use_weather
        
        # Branch 1: Temporal Transformer-LSTM Backbone (processes node data flatly)
        self.temporal_backbone = TransformerLSTMModel(
            input_dim=traffic_feat_dim, 
            output_dim=output_dim, 
            seq_len=seq_len, 
            **transformer_kwargs
        )
        
        # Branch 2: Topological Graph Convolution (GNN) Branch
        self.graph_conv1 = GCNConv(traffic_feat_dim, graph_hidden_dim)
        self.graph_conv2 = GCNConv(graph_hidden_dim, graph_hidden_dim)
        self.graph_fc = nn.Linear(graph_hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
        # Branch 3: Weather/Global Condition Branch (Linear Context Encoder)
        if self.use_weather:
            self.weather_encoder = nn.Sequential(
                nn.Linear(weather_feat_dim * seq_len, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 32)
            )
            weather_out_dim = 32
        else:
            weather_out_dim = 0
        total_fused_dim = (num_nodes * output_dim) + (num_nodes * output_dim) + weather_out_dim
        
        self.fusion_head = nn.Sequential(
            nn.Linear(total_fused_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, num_nodes * output_dim)
        )

    def forward(self, traffic_x, edge_index, weather_x=None):
        batch_size, N, S, F = traffic_x.shape
        
        # --- Branch 1: Transformer-LSTM Feature Extraction ---
        t_reshaped = traffic_x.view(batch_size * N, S, F)
        trans_out = self.temporal_backbone(t_reshaped)
        trans_flat = trans_out.view(batch_size, -1)
        
        # --- Branch 2: Spatial Graph Messaging Pass ---
        spatial_features = traffic_x.mean(dim=2)
        
        graph_outs = []
        for b in range(batch_size):
            g_x = spatial_features[b] # (N, F)
            g_hidden = self.relu(self.graph_conv1(g_x, edge_index))
            g_hidden = self.dropout(g_hidden)
            g_hidden = self.relu(self.graph_conv2(g_hidden, edge_index))
            g_out = self.graph_fc(g_hidden)
            graph_outs.append(g_out.view(-1))
            
        graph_flat = torch.stack(graph_outs, dim=0)
        
        # --- Branch 3: Weather Branch Context (Toggled Condition) ---
        if self.use_weather and weather_x is not None:
            w_flat = weather_x.reshape(batch_size, -1)
            weather_repr = self.weather_encoder(w_flat)
            fused_vector = torch.cat([trans_flat, graph_flat, weather_repr], dim=-1)
        else:
            fused_vector = torch.cat([trans_flat, graph_flat], dim=-1)
            
        predictions = self.fusion_head(fused_vector)
        return predictions.view(batch_size, N, self.output_dim)

In [123]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset

class SpatioTemporalDataset(Dataset):
    def __init__(self, traffic_data, weather_data, target_data, lookback, forecast_steps, use_weather=True):
        """
        traffic_data: Tensor of shape (Total_Time_Steps, Num_Nodes, Traffic_Feat_Dim)
        weather_data: Tensor of shape (Total_Time_Steps, Weather_Feat_Dim)
        target_data: Tensor of shape (Total_Time_Steps,) containing the PM2.5 target series
        """
        self.traffic_data = traffic_data
        self.weather_data = weather_data
        self.target_data = target_data
        self.lookback = lookback
        self.forecast_steps = forecast_steps
        self.use_weather = use_weather
        
        self.num_samples = min(
            len(traffic_data),
            len(weather_data) if weather_data is not None else len(traffic_data),
            len(target_data) if target_data is not None else len(traffic_data)
        ) - lookback - forecast_steps + 1
        self.num_samples = max(self.num_samples, 0)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start_lookback = idx
        end_lookback = idx + self.lookback
        
        traffic_x = self.traffic_data[start_lookback:end_lookback].transpose(0, 1)
        
        start_forecast = end_lookback
        end_forecast = end_lookback + self.forecast_steps
        target_window = self.target_data[start_forecast:end_forecast]
        y_true = target_window.unsqueeze(0).repeat(traffic_x.size(0), 1)
        
        if self.use_weather and self.weather_data is not None:
            weather_x = self.weather_data[start_lookback:end_lookback]
            return traffic_x, weather_x, y_true
        else:
            return traffic_x, y_true

# =========================================================
# INITIALIZATION EXAMPLE USING DATA FROM YOUR WORKSPACE
# =========================================================

LOOKBACK = 24*3
FORECAST = 24

traffic_tensor = torch.tensor(traffic_time_major, dtype=torch.float32)
weather_tensor = torch.tensor(scaled_temporal, dtype=torch.float32) if ADDITIONAL_BRANCH else None
target_tensor = torch.tensor(target_series, dtype=torch.float32)



# Step B: Instantiate Training + Validation sets from the aligned full dataset
full_dataset = SpatioTemporalDataset(
    traffic_data=traffic_tensor,
    weather_data=weather_tensor,
    target_data=target_tensor,
    lookback=LOOKBACK,
    forecast_steps=FORECAST,
    use_weather=ADDITIONAL_BRANCH
)

num_samples = len(full_dataset)

split = int(num_samples * 0.8)
train_dataset = Subset(full_dataset, list(range(split)))
test_dataset = Subset(full_dataset, list(range(split, num_samples)))

# Step C: Construct DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [124]:
import torch.optim as optim

def execute_model_training(model, train_loader, test_loader, edge_index, epochs, lr, device):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    edge_index = edge_index.to(device)
    
    print(f"Starting Training Matrix on device: {device}")
    print(f"Weather Context Branch Status: {'ENABLED' if model.use_weather else 'DISABLED'}\n")
    
    for epoch in range(1, epochs + 1):
        # --- TRAINING CYCLE ---
        model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            if model.use_weather:
                traffic_x, weather_x, y_true = batch
                traffic_x, weather_x, y_true = traffic_x.to(device), weather_x.to(device), y_true.to(device)
                predictions = model(traffic_x, edge_index, weather_x)
            else:
                traffic_x, y_true = batch
                traffic_x, y_true = traffic_x.to(device), y_true.to(device)
                predictions = model(traffic_x, edge_index, weather_x=None)
                
            loss = criterion(predictions, y_true)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * traffic_x.size(0)
            
        epoch_train_loss = train_loss / len(train_loader.dataset)
        
        # --- TEST ACCURACY EVALUATION ---
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                if model.use_weather:
                    traffic_x, weather_x, y_true = batch
                    traffic_x, weather_x, y_true = traffic_x.to(device), weather_x.to(device), y_true.to(device)
                    predictions = model(traffic_x, edge_index, weather_x)
                else:
                    traffic_x, y_true = batch
                    traffic_x, y_true = traffic_x.to(device), y_true.to(device)
                    predictions = model(traffic_x, edge_index, weather_x=None)
                    
                loss = criterion(predictions, y_true)
                test_loss += loss.item() * traffic_x.size(0)
                
        epoch_test_loss = test_loss / len(test_loader.dataset)
        
        print(f"Epoch [{epoch:02d}/{epochs:02d}] -> Train Loss (MSE): {epoch_train_loss:.5f} | Test Loss (MSE): {epoch_test_loss:.5f}")

In [125]:
# 1. Configuration Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
node_labels = list(traffic_dic.keys())
edge_index = pyg_dataset.edge_index

transformer_hyperparams = {
    "d_model": 64,
    "nhead": 4,
    "num_transformer_layers": 2,
    "hidden_dim": 64,
    "num_lstm_layers": 2,
    "dropout": 0.3
}

# 2. Instantiate TriBranch Model 
tri_model = TriBranchSpatioTemporalModel(
    num_nodes=len(node_labels),
    traffic_feat_dim=traffic_tensor.shape[-1],
    weather_feat_dim=weather_tensor.shape[-1] if ADDITIONAL_BRANCH else 0,
    seq_len=LOOKBACK,
    output_dim=FORECAST,
    transformer_kwargs=transformer_hyperparams,
    use_weather=ADDITIONAL_BRANCH
).to(device)

# 3. Train Model
execute_model_training(
    model=tri_model,
    train_loader=train_loader,
    test_loader=test_loader,
    edge_index=edge_index,
    epochs=50,
    lr=0.001,
    device=device
)

Starting Training Matrix on device: cuda
Weather Context Branch Status: DISABLED

Epoch [01/50] -> Train Loss (MSE): 65.34172 | Test Loss (MSE): 27.17966
Epoch [02/50] -> Train Loss (MSE): 58.08757 | Test Loss (MSE): 27.03452
Epoch [03/50] -> Train Loss (MSE): 57.28994 | Test Loss (MSE): 28.67942
Epoch [04/50] -> Train Loss (MSE): 56.40985 | Test Loss (MSE): 31.85605
Epoch [05/50] -> Train Loss (MSE): 55.39058 | Test Loss (MSE): 28.63311
Epoch [06/50] -> Train Loss (MSE): 55.59304 | Test Loss (MSE): 28.80205
Epoch [07/50] -> Train Loss (MSE): 55.37812 | Test Loss (MSE): 29.59617
Epoch [08/50] -> Train Loss (MSE): 55.10804 | Test Loss (MSE): 28.56236
Epoch [09/50] -> Train Loss (MSE): 55.17460 | Test Loss (MSE): 29.78474
Epoch [10/50] -> Train Loss (MSE): 54.62668 | Test Loss (MSE): 30.01002
Epoch [11/50] -> Train Loss (MSE): 54.27459 | Test Loss (MSE): 28.99316
Epoch [12/50] -> Train Loss (MSE): 54.25880 | Test Loss (MSE): 27.45054
Epoch [13/50] -> Train Loss (MSE): 54.18746 | Test Los

In [126]:
def inverse_transform_target(scaled_data, scaler, target_idx, n_features):
    flat_data = scaled_data.flatten()
    dummy = np.zeros((len(flat_data), n_features))
    dummy[:, target_idx] = flat_data
    unscaled_dummy = scaler.inverse_transform(dummy)
    return unscaled_dummy[:, target_idx].reshape(scaled_data.shape)

In [127]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_tri_branch_global(model, test_loader, edge_index, device):
    model.eval()
    edge_index = edge_index.to(device)

    all_preds = []
    all_trues = []

    with torch.no_grad():
        for batch in test_loader:
            if model.use_weather:
                traffic_x, weather_x, y_true = batch
                traffic_x, weather_x = traffic_x.to(device), weather_x.to(device)
                preds = model(traffic_x, edge_index, weather_x)
            else:
                traffic_x, y_true = batch
                traffic_x = traffic_x.to(device)
                preds = model(traffic_x, edge_index, None)

            # collapse node dimension → PM2.5 is not node-specific
            preds_mean = preds.mean(axis=1)      # (batch, forecast)
            trues_mean = y_true.mean(axis=1)     # (batch, forecast)

            all_preds.append(preds_mean.cpu().numpy())
            all_trues.append(trues_mean.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)

    # evaluate on the original PM2.5 scale directly
    global_mae = mean_absolute_error(all_trues.flatten(), all_preds.flatten())
    global_rmse = np.sqrt(mean_squared_error(all_trues.flatten(), all_preds.flatten()))
    global_r2 = r2_score(all_trues.flatten(), all_preds.flatten())

    ground_truth_mean = np.mean(all_trues)
    mae_percentage = global_mae / ground_truth_mean if ground_truth_mean != 0 else np.nan
    rmse_percentage = global_rmse / ground_truth_mean if ground_truth_mean != 0 else np.nan

    print("\n=== Final Tri‑Branch PM2.5 Forecast Results ===")
    print(f"Global R² Score     : {global_r2:.4f}")
    print(f"Global MAE          : {global_mae:.4f} μg/m³")
    print(f"Global RMSE         : {global_rmse:.4f} μg/m³")
    print(f"MAE% (Normalized)   : {mae_percentage:.2f}")
    print(f"RMSE% (Normalized)  : {rmse_percentage:.2f}")

    return all_trues, all_preds


In [128]:
true_ground_truth_all, true_predictions_all = evaluate_tri_branch_global(
    model=tri_model,
    test_loader=test_loader,
    edge_index=edge_index,
    device=device
)


=== Final Tri‑Branch PM2.5 Forecast Results ===
Global R² Score     : -0.0380
Global MAE          : 3.6474 μg/m³
Global RMSE         : 5.3148 μg/m³
MAE% (Normalized)   : 0.40
RMSE% (Normalized)  : 0.58


In [129]:
def plot_tri_branch_horizons(true_ground_truth_all, true_predictions_all, df, INPUT_WINDOW, OUTPUT_WINDOW):
    sample_indices = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500]
    num_samples = len(sample_indices)

    fig = go.Figure()
    buttons = []

    for idx, sample_idx in enumerate(sample_indices):
        true_actual = true_ground_truth_all[sample_idx]
        true_pred = true_predictions_all[sample_idx]

        start_timestamp_idx = INPUT_WINDOW + sample_idx
        window_timestamps = df.index[start_timestamp_idx : start_timestamp_idx + OUTPUT_WINDOW]

        time_labels = [str(t).split()[1][:5] for t in window_timestamps]
        date_str = str(window_timestamps[0]).split()[0]

        fig.add_trace(go.Scatter(
            x=time_labels, y=true_actual,
            mode='lines+markers',
            name=f'Actual PM2.5 ({date_str})',
            line=dict(color='seagreen', width=3),
            visible=(idx == 0)
        ))

        fig.add_trace(go.Scatter(
            x=time_labels, y=true_pred,
            mode='lines+markers',
            name=f'Tri‑Branch Forecast ({date_str})',
            line=dict(color='darkorange', width=2, dash='dash'),
            visible=(idx == 0)
        ))

        visibility_mask = [False] * (num_samples * 2)
        visibility_mask[idx * 2] = True
        visibility_mask[idx * 2 + 1] = True

        buttons.append(dict(
            label=f"Horizon: {date_str}",
            method="update",
            args=[
                {"visible": visibility_mask},
                {"title": f"<b>Tri‑Branch Performance Horizon: {date_str}</b>"}
            ]
        ))

    initial_date_str = str(df.index[INPUT_WINDOW]).split()[0]

    fig.update_layout(
        title=dict(
            text=f'<b>Tri‑Branch Performance Horizon: {initial_date_str}</b>',
            x=0.5,
            font=dict(size=16)
        ),
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.0,
                xanchor="left",
                y=1.15,
                yanchor="top"
            )
        ],
        xaxis=dict(title='Hour of Day (24h Forecast Horizon)', showgrid=True, gridcolor='lightgray'),
        yaxis=dict(title='PM2.5 Concentration (μg/m³)', showgrid=True, gridcolor='lightgray'),
        template='plotly_white',
        hovermode='x unified',
        height=550,
        width=900,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.show()


In [130]:
plot_tri_branch_horizons(
    true_ground_truth_all=true_ground_truth_all,
    true_predictions_all=true_predictions_all,
    df=temporal_df,
    INPUT_WINDOW=LOOKBACK,
    OUTPUT_WINDOW=FORECAST
)